# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane (Refresh / Content Opportunity Scoring) is primarily a **ranking / scoring** task, with a **classification** model underneath it.

The end product a reviewer sees is a *ranked list* of pages, ordered by how urgently they should be reviewed — that's ranking. But the way I'll build that ranking, at least for this phase, is by training a binary classifier ("is this page declining or not") and using its predicted probability as the score to sort by. This is the same pattern used in notebooks 01 and 02: a decision tree's `predict_proba` output became the ranking signal, and I evaluated it with Precision@K, a ranking-style metric, not plain classification accuracy.

It is not clustering, because I'm not trying to discover unlabeled groups of similar pages — I already have a target concept (declining vs. not) I want to predict and rank by.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**What I'd predict:** the probability that a page is currently declining in search performance, used to rank pages for review.

**Where the label comes from:** it is a **proxy**, not a directly observed "ground truth" outcome. The starter data defines it as `trend_direction == "down"` — a bucket that some upstream process already computed from a raw percentage change (`trend_pct`). I'm not observing a reviewer's real decision or a verified future outcome; I'm reusing a pre-computed label as a stand-in for "this page needs attention."

This matters because a proxy label can be wrong in ways I can't easily check — e.g. a page could show `trend_direction == "down"` from a small blip, not a real problem. A stronger version of this task (a stretch goal for later weeks) would define the label as a genuinely future outcome — e.g. "this page's impressions drop by X% over the next 30 days" — using only features known before that window, so the model has to actually predict the future rather than describe the present.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Metric: Precision@50** (and Precision@20 as a secondary check).

I'm choosing this because the real-world action is bounded by reviewer capacity — a reviewer can only look at a fixed number of pages in a given period, not the whole list. Precision@K directly answers the question that matters for the action: "of the top K pages I hand the reviewer, how many were actually worth their time?" This is the same metric the starter pipeline and my own notebooks already used, so it's consistent with prior work and I can defend it with numbers I've already produced.

I'm not using plain accuracy, because the declining rate is roughly balanced-but-not-extreme in this sample (~54%), so accuracy would be a weak signal either way — and more importantly, accuracy treats every page equally, while my actual use case only cares about the pages at the very top of the ranked list.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*One row = one page.* Loaded and shown below.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define the proxy target the same way notebooks 01/02 did
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# The lane's slice: one row = one page, with the fields relevant to review-priority scoring
cols = ["content_age_days", "days_since_last_update", "impressions_90d",
        "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]
lane_df = df[cols].copy()

print(f"Rows (= pages): {lane_df.shape[0]}")
print(f"Columns: {list(lane_df.columns)}\n")
print("First 5 rows - each row is one page:")
lane_df.head()


Rows (= pages): 30000
Columns: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'trend_direction', 'is_declining_label']

First 5 rows - each row is one page:


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,down,1
1,445,25,15320,20.3,0.05,2481.0,down,1
2,141,20,12581,36.5,0.09,3515.0,down,1
3,463,22,11751,6.2,0.49,NaN,stable,0
4,263,14,19140,44.0,0.13,2803.0,down,1


In [5]:
# Sketch of the target column: distribution of the proxy label
print("Target column: is_declining_label (1 = declining/proxy-flagged, 0 = not)")
print(lane_df["is_declining_label"].value_counts(normalize=True).round(3))

Target column: is_declining_label (1 = declining/proxy-flagged, 0 = not)
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 5. Why ML beats a fixed rule here

A fixed rule like "stale AND visible" only uses two columns with hard thresholds (180 days, 500 impressions), and treats every page that crosses those thresholds identically regardless of degree. Real pages don't fail cleanly along one or two dimensions — a page could be moderately stale but have a sharply rising CTR (probably fine), or be fairly fresh but sitting at a terrible average position with high impressions (probably a real opportunity). A hand rule can't weigh six interacting signals (age, staleness, impressions, position, CTR, word count) against each other, or learn that the *combination* of two moderate signals matters more than one extreme signal.

This is exactly what showed up empirically in notebook 02: the depth-2 decision tree, which can only ask 3 yes/no questions total, already beat the hand rule at Precision@50 (0.600 vs whatever the hand rule scored), and the starter pipeline's full model beat a baseline rule by roughly 3x (0.740 vs 0.240). That gap is evidence the pattern is genuinely multi-dimensional and interacting, not something a human-written if/else can capture as well as a model that learns thresholds and interactions directly from the data.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.